# Config-Driven Binary CNN — GPU Training
10-epoch GPU training driven by `experiments/configs/binary_baseline.yaml`.  
No checkpoint saving. No external logging.

In [ ]:
# ─── Section 1: Imports and Config ────────────────────────────────────────────
import os
import sys
import time

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import yaml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from qcore.data.registry import get_dataset
from qcore.data.torch_adapter import TorchDatasetAdapter
from qcore.models.cnn import build_model

CONFIG_PATH = os.path.join(PROJECT_ROOT, 'experiments', 'configs', 'binary_baseline.yaml')
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', f'Expected CUDA, got: {DEVICE}. Check GPU runtime.'

print(f'Device : {DEVICE}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Config : {config}')

In [ ]:
# ─── Section 2: Load Datasets and DataLoaders ─────────────────────────────────
print('Loading datasets ...')

dataset_name = config['dataset']['name']
batch_size   = config['training']['batch_size']

train_ds = get_dataset(dataset_name, 'train')
val_ds   = get_dataset(dataset_name, 'val')
test_ds  = get_dataset(dataset_name, 'test')

train_adapted = TorchDatasetAdapter(train_ds)
val_adapted   = TorchDatasetAdapter(val_ds)
test_adapted  = TorchDatasetAdapter(test_ds)

train_loader = DataLoader(train_adapted, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_adapted,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_adapted,  batch_size=batch_size, shuffle=False)

print(f'Train samples : {len(train_ds)}')
print(f'Val   samples : {len(val_ds)}')
print(f'Test  samples : {len(test_ds)}')

# ── Class weights (optional) ──────────────────────────────────────────────────
if config['training']['class_weights']:
    all_train_labels = [train_adapted[i][1].item() for i in range(len(train_adapted))]
    all_train_labels_t = torch.tensor(all_train_labels, dtype=torch.long)
    num_classes = config['dataset']['num_classes']
    class_counts = torch.zeros(num_classes)
    for c in range(num_classes):
        class_counts[c] = (all_train_labels_t == c).sum().float()
    class_weights = class_counts.sum() / (num_classes * class_counts)
    class_weights = class_weights.to(DEVICE)
    print(f'Class weights : {class_weights.cpu().tolist()}')
else:
    class_weights = None
    print('Class weights : disabled')

In [ ]:
# ─── Section 3: Visualize Sample Batch ────────────────────────────────────────
label_map = train_ds.label_map

x_batch, y_batch = next(iter(train_loader))
print(f'Batch shape : {x_batch.shape}  dtype={x_batch.dtype}')

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.flat, range(8)):
    img_np = x_batch[i].cpu().numpy().squeeze()
    lbl    = int(y_batch[i].item())
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax.set_title(label_map.get(lbl, str(lbl)), fontsize=8)
    ax.axis('off')

plt.suptitle(f'{dataset_name} — sample train batch (config-driven)', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'cfg_pneumoniamnist_sample_batch.png'), dpi=120)
plt.show()
print('Sample batch plot saved.')

In [ ]:
# ─── Section 4: Build Model ───────────────────────────────────────────────────
model_config = {}
for k, v in config['model'].items():
    model_config[k] = v
for k, v in config['dataset'].items():
    model_config[k] = v

model = build_model(model_config).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters : {n_params}')

In [ ]:
# ─── Section 5: Train Loop ─────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['lr'])
loss_fn   = nn.CrossEntropyLoss(weight=class_weights)

train_losses = []
val_losses   = []
train_accs   = []
val_accs     = []
epoch_times  = []

epochs = config['training']['epochs']

for epoch in range(1, epochs + 1):
    t0 = time.time()

    # ── Train pass ────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    n_correct    = 0
    n_total      = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds         = logits.argmax(dim=1)
        n_correct    += (preds == yb).sum().item()
        n_total      += xb.size(0)

    epoch_train_loss = running_loss / n_total
    epoch_train_acc  = n_correct / n_total

    # ── Val pass ──────────────────────────────────────────────────────────────
    model.eval()
    val_running_loss = 0.0
    val_n_correct    = 0
    val_n_total      = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            loss    = loss_fn(logits, yb)
            val_running_loss += loss.item() * xb.size(0)
            preds             = logits.argmax(dim=1)
            val_n_correct    += (preds == yb).sum().item()
            val_n_total      += xb.size(0)

    epoch_val_loss = val_running_loss / val_n_total
    epoch_val_acc  = val_n_correct / val_n_total

    epoch_time = time.time() - t0
    epoch_times.append(epoch_time)
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(epoch_train_acc)
    val_accs.append(epoch_val_acc)

    print(
        f'Epoch {epoch}/{epochs} | '
        f'train_loss={epoch_train_loss:.4f} | '
        f'val_loss={epoch_val_loss:.4f} | '
        f'train_acc={epoch_train_acc:.4f} | '
        f'val_acc={epoch_val_acc:.4f} | '
        f'{epoch_time:.2f}s'
    )

print('Training complete.')

In [ ]:
# ─── Section 6: Plot Loss Curve ───────────────────────────────────────────────
epochs_axis = list(range(1, epochs + 1))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_axis, train_losses, marker='o', label='Train loss')
ax.plot(epochs_axis, val_losses,   marker='s', label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title(f'{dataset_name} config-driven — loss curves')
ax.legend()
ax.set_xticks(epochs_axis)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'cfg_pneumoniamnist_loss_curve.png'), dpi=120)
plt.show()
print('Loss curve saved.')

In [ ]:
# ─── Section 7: Plot Accuracy Curve ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_axis, [a * 100 for a in train_accs], marker='o', label='Train accuracy')
ax.plot(epochs_axis, [a * 100 for a in val_accs],   marker='s', label='Val accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{dataset_name} config-driven — accuracy curves')
ax.legend()
ax.set_xticks(epochs_axis)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'cfg_pneumoniamnist_acc_curve.png'), dpi=120)
plt.show()
print('Accuracy curve saved.')

In [ ]:
# ─── Section 8: Test Evaluation + Inference Latency ───────────────────────────
model.eval()
batch_latencies = []
all_preds       = []
all_labels      = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        t0 = time.time()
        logits = model(xb)
        batch_latencies.append((time.time() - t0) * 1000)
        preds = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(yb.tolist())

test_correct    = sum(p == l for p, l in zip(all_preds, all_labels))
test_accuracy   = test_correct / len(all_labels)
mean_latency_ms = sum(batch_latencies) / len(batch_latencies)

print(f'Test accuracy          : {test_accuracy * 100:.2f}%  ({test_correct}/{len(all_labels)})')
print(f'Mean inference latency : {mean_latency_ms:.3f} ms per batch')

In [ ]:
# ─── Section 9: Confusion Matrix ──────────────────────────────────────────────
class_names = [label_map.get(i, str(i)) for i in sorted(label_map.keys())]
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'{dataset_name} config-driven — test confusion matrix')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'cfg_pneumoniamnist_confusion_matrix.png'), dpi=120)
plt.show()
print('Confusion matrix saved.')

In [ ]:
# ─── Section 10: Sample Predictions ───────────────────────────────────────────
model.eval()
x_test_batch, y_test_batch = next(iter(test_loader))

with torch.no_grad():
    test_logits = model(x_test_batch.to(DEVICE))
    test_preds  = test_logits.argmax(dim=1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.flat, range(8)):
    img_np    = x_test_batch[i].cpu().numpy().squeeze()
    pred      = int(test_preds[i].item())
    true      = int(y_test_batch[i].item())
    pred_name = label_map.get(pred, str(pred))
    true_name = label_map.get(true, str(true))
    color     = 'green' if pred == true else 'red'
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'P:{pred_name}\nT:{true_name}', fontsize=7, color=color)
    ax.axis('off')

plt.suptitle(f'{dataset_name} config-driven — sample test predictions (green=correct, red=wrong)', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'cfg_pneumoniamnist_sample_preds.png'), dpi=120)
plt.show()
print('Sample predictions plot saved.')

In [ ]:
# ─── Section 11: Summary Block ────────────────────────────────────────────────
GO_THRESHOLD    = 0.70
mean_epoch_time = sum(epoch_times) / len(epoch_times)
verdict         = 'GO' if test_accuracy >= GO_THRESHOLD else 'NO-GO'
n_params        = sum(p.numel() for p in model.parameters())

print('=== CONFIG-DRIVEN BASELINE SUMMARY ===')
print(f'Device             : {DEVICE}')
print(f'GPU                : {torch.cuda.get_device_name(0)}')
print(f'Config             : experiments/configs/binary_baseline.yaml')
print(f'Conv channels      : {config["model"]["conv_channels"]}')
print(f'BatchNorm          : {config["model"]["use_batchnorm"]}')
print(f'Dropout            : {config["model"]["dropout"]}')
print(f'Class weights      : {config["training"]["class_weights"]}')
print(f'Total parameters   : {n_params}')
print(f'Epochs             : {epochs}')
print(f'Mean epoch time    : {mean_epoch_time:.2f}s')
print(f'Final train accuracy: {train_accs[-1] * 100:.2f}%')
print(f'Final val accuracy  : {val_accs[-1] * 100:.2f}%')
print(f'Test accuracy       : {test_accuracy * 100:.2f}%')
print(f'Mean inference latency: {mean_latency_ms:.2f}ms per batch')
print(f'{verdict}')